# Building AI Chat with FastAPI - From Stateless to Stateful

**Module 7 · Lesson 7.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Chat Session Manager - Track Who Said What
- System Prompt Patterns - Give Your Bot a Soul
- Context Window Strategies - When History Gets Too Long
- Persistent Storage - Survive Server Restarts
- Project Structure & Dependency Injection - Production FastAPI Layout
- SSE Streaming Endpoint - fastapi.sse + Async Generators

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The Demo That Forgot Its Customer

A support-bot team spent three weeks polishing answers. Launch demo, CEO at the keyboard:

**📝 `demo_transcript.txt`** - *Intro*

In [ ]:
# Output: the launch demo, reconstructed from the meeting notes
#
# CEO:  "My order #4521 arrived with a cracked screen."
# Bot:  "Sorry to hear that! For damaged items we offer replacement or
#        refund. Could you share a photo of the damage?"
# CEO:  "OK. So which one should I pick?"
# Bot:  "Which one of WHAT? Hi! How can I help you today?"
#
# ...silence in the room.
#
# Post-mortem, one line of backend code:
#   answer = llm(new_message)          # sends ONLY the newest message
# The team had tested 400 single questions. Nobody tested a CONVERSATION.

Every piece of this lesson exists to prevent that meeting: a session store so each user has a transcript (step 1), a system prompt so the bot has a job description (step 2), trimming so long transcripts still fit and don't bankrupt you (step 3), Redis so a deploy doesn't wipe every conversation (step 4), and a FastAPI service that ships it (steps 5-10). And here's the embarrassing part: an earlier version of *this very lesson's* project code had the same bug in its streaming endpoint - it computed the trimmed history, then sent only the new message. You'll see the guard rails we added so you never repeat it.

> 📦 **Info**
>
> 🧩 Before you start
> - **Lesson 7.1** - you can create an API client and call an LLM with retries. We reuse that muscle everywhere here.
> - **Lesson 7.2** - you know what SSE frames look like on the wire and why proxies buffer streams. Step 6 builds the FastAPI side of that pipe.
> - **Lesson 7.3** - you can read a token bill. Step 3 is where conversation history becomes the bill's biggest line item.
> - **Tools** - Python 3.12+, `pip install fastapi uvicorn google-genai redis`. Redis itself is optional until step 4 (use Docker: `docker run -p 6379:6379 redis`).

## The Core Problem: LLMs Forget Instantly

> 🐘 **Analogy**
>
> **Think of a goldfish customer support agent.** Every 3 seconds, they forget everything. Customer says "I ordered a laptop." Agent: "Great!" Customer: "The screen is broken." Agent: "What laptop? Who are you?" That's what an LLM does without memory - every API call is a fresh start with zero context.
> To make the AI "remember," YOU send the entire conversation history with every request. Turn 1: send message 1. Turn 5: send messages 1, 2, 3, 4, AND 5. Turn 50: send all fifty messages. **The illusion of memory is engineered by your code, not the model.**

> 📦 **Info**
>
> ⚠️ Misconception check: "But I passed a session ID - the model remembers my session"
> No. The session ID is bookkeeping *your server* understands; the provider API is stateless and sees only the `messages` array inside THIS request. If a message is not in that array, for the model it never happened. The same trap in disguise: "Redis gives the AI long-term memory." Redis persists *your transcript*; the model still reads only what you place in the context window on each call. Memory is not a feature you enable - it is a payload you assemble, every single turn.

> ℹ️ **Info**
>
> What you'll build
> - **Chat session manager** - track conversations per user with proper message format
> - **System prompt patterns** - persona, rules, and dynamic context injection
> - **Context window strategies** - what to do when conversation exceeds the limit
> - **Persistent storage** - save and resume conversations across server restarts
> - **Project** - production chat API with sessions, memory, streaming, and persistence

## 60-Second Warm-Up: What You Already Know

Three recalls from earlier lessons - each one is load-bearing today. Click each card to check yourself.

## The Chat Engine: Sessions, Prompts, Memory, Persistence

---

## 🎯 Concept 1: Chat Session Manager - Track Who Said What

### Chat Session Manager - Track Who Said What

Each user gets their own conversation. Messages stay in order. Sessions are isolated.

Before FastAPI, before Redis, before anything with a port number: the data structure everything else in this lesson stands on. Three small classes - a message, a conversation, a registry of conversations. Get the shape right here and every later step is a decorator on top of it.

> ⚖️ **Analogy**
>
> **A courtroom stenographer.** Every case (session) has its own transcript. Every utterance is logged with WHO said it - counsel, witness, judge (the `role` field) - in strict order. When the case resumes next week, the judge is read the transcript aloud; the judge remembers nothing on their own. The transcript IS the memory, and the stenographer (your `ChatSession`) owns it.
> **Where this analogy breaks down:** a court transcript is legally immutable. Yours is not - and that's a feature. In step 3 you will deliberately trim, summarize, and rewrite the copy you read back to the judge, and the judge will never know.

**📝 `part1_session_manager.py`** - *language-python*

In [ ]:
# =============================================
# CHAT SESSION MANAGER
# Each user gets their own conversation history.
# Messages are stored in the format the LLM expects.
# =============================================

import uuid
from datetime import datetime
from dataclasses import dataclass, field

@dataclass
class Message:
    role: str           # "user", "assistant", or "system"
    content: str
    timestamp: float = field(default_factory=lambda: datetime.now().timestamp())
    tokens: int = 0

class ChatSession:
    """One conversation between one user and the AI."""
    
    def __init__(self, session_id: str = None, system_prompt: str = ""):
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.messages: list[Message] = []
        self.created_at = datetime.now()
        
        if system_prompt:
            self.messages.append(Message(role="system", content=system_prompt))
    
    def add_user_message(self, content: str):
        self.messages.append(Message(role="user", content=content))
    
    def add_assistant_message(self, content: str):
        self.messages.append(Message(role="assistant", content=content))
    
    def to_api_format(self) -> list[dict]:
        """Convert to the format LLM APIs expect."""
        return [{"role": m.role, "content": m.content} for m in self.messages]
    
    def turn_count(self) -> int:
        return sum(1 for m in self.messages if m.role == "user")

class SessionStore:
    """Manage multiple chat sessions (one per user/conversation)."""
    
    def __init__(self):
        self.sessions: dict[str, ChatSession] = {}
    
    def create(self, system_prompt: str = "") -> ChatSession:
        session = ChatSession(system_prompt=system_prompt)
        self.sessions[session.session_id] = session
        return session
    
    def get(self, session_id: str) -> ChatSession | None:
        return self.sessions.get(session_id)
    
    def delete(self, session_id: str):
        self.sessions.pop(session_id, None)

# ── Demo ──
store = SessionStore()
session = store.create(system_prompt="You are a helpful AI tutor for the Netsetos course.")

session.add_user_message("What is RAG?")
session.add_assistant_message("RAG stands for Retrieval-Augmented Generation...")
session.add_user_message("How does the retrieval part work?")

print(f"Session: {session.session_id}")
print(f"Turns: {session.turn_count()}")
print(f"\nAPI format (what the LLM sees):")
for msg in session.to_api_format():
    print(f"  [{msg['role']:9s}] {msg['content'][:60]}...")

# Output:
#   Session: a3f8c2e1
#   Turns: 2
#   API format (what the LLM sees):
#     [system   ] You are a helpful AI tutor for the Netsetos course....
#     [user     ] What is RAG?...
#     [assistant] RAG stands for Retrieval-Augmented Generation......
#     [user     ] How does the retrieval part work?...

#### 💡 What just happened

What just happened?
  We built the foundation: a `Message` dataclass (role + content + timestamp), a `ChatSession` that tracks one conversation with `to_api_format()` for sending to any LLM, and a `SessionStore` that manages multiple sessions by ID. Every chat app starts here - before context management, before persistence, before anything else. **When to use this vs a framework:** hand-roll it (as here) when you want zero dependencies and full control; reach for a framework’s memory classes instead when you are already inside its agent stack - the tradeoff is transparency for convenience.

---

## 🎯 Concept 2: System Prompt Patterns - Give Your Bot a Soul

### System Prompt Patterns - Give Your Bot a Soul

The system prompt defines WHO the AI is, HOW it behaves, and WHAT it can/can't do.

Step 1's `messages` array carries three roles. You've tracked two of them - `user` and `assistant`. The third, `system`, is the one your users never see and your product lives or dies by: it turns a generic model into YOUR support agent, YOUR tutor, YOUR brand voice. It is also the one message that must survive every trim in step 3.

> 🎭 **Analogy**
>
> **Think of an employee handbook.** When someone joins a company, they get a handbook: your title is X, your responsibilities are Y, company tone is Z, these topics are off-limits. The system prompt is the AI's employee handbook - it defines personality, rules, knowledge boundaries, and response style. Without it, the AI is a generic assistant. With a great one, it feels like a custom-built product.
> **Where this analogy breaks down:** handbooks don't enforce themselves. An employee sweet-talked by a customer may still break policy - and a model sweet-talked by a crafted message will too. That is prompt injection (OWASP LLM01), and it is why step 9 treats user input as data to be quarantined, never as instructions. The handbook is product design; it is not security.

**📝 `part2_system_prompts.py`** - *language-python*

In [ ]:
# =============================================
# SYSTEM PROMPT PATTERNS for different use cases.
# Copy, customize, and use in your projects.
# =============================================

class SystemPromptBuilder:
    """Build structured system prompts from components."""
    
    def __init__(self):
        self.sections = {}
    
    def set_role(self, role: str):
        self.sections["role"] = f"# ROLE\n{role}"
        return self
    
    def set_rules(self, rules: list[str]):
        numbered = "\n".join([f"{i+1}. {r}" for i, r in enumerate(rules)])
        self.sections["rules"] = f"# RULES (never violate)\n{numbered}"
        return self
    
    def set_style(self, style: str):
        self.sections["style"] = f"# RESPONSE STYLE\n{style}"
        return self
    
    def set_knowledge(self, knowledge: str):
        self.sections["knowledge"] = f"# KNOWLEDGE BASE\n{knowledge}"
        return self
    
    def set_examples(self, examples: list[tuple[str, str]]):
        ex_text = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in examples])
        self.sections["examples"] = f"# EXAMPLE INTERACTIONS\n{ex_text}"
        return self
    
    def build(self) -> str:
        return "\n\n".join(self.sections.values())

# ── Pattern 1: Customer Support Bot ──
support_prompt = (SystemPromptBuilder()
    .set_role("You are Asha, a customer support agent for Netsetos EdTech platform.")
    .set_rules([
        "Only answer questions about Netsetos courses, pricing, and technical issues.",
        "For billing issues, collect the order ID and escalate to human support.",
        "Never share internal policies, discount codes, or employee information.",
        "If asked about competitors, say 'I can only help with Netsetos questions.'",
        "Always be polite, even if the user is frustrated.",
    ])
    .set_style("Respond in 2-3 sentences. Use a warm, professional tone. Use the student's name if known.")
    .set_examples([
        ("How much is the GenAI course?", "The GenAI course is ₹29,999. It includes 18 modules, about 150 hours of content, and 6 months of access. Would you like me to help with enrollment?"),
        ("Your course is too expensive", "I understand budget is important! We offer EMI options starting at ₹2,500/month. We also have early-bird discounts - want me to check if any are active?"),
    ])
    .build()
)

# ── Pattern 2: Coding Tutor ──
tutor_prompt = (SystemPromptBuilder()
    .set_role("You are a Python coding tutor for beginners. You teach by showing small code examples first, then explaining.")
    .set_rules([
        "Always show code FIRST, then explain what it does.",
        "Keep code examples under 15 lines.",
        "Use Indian analogies when helpful (cricket, chai, Bollywood).",
        "If the student makes a mistake, don't give the answer directly - ask guiding questions.",
        "End every response with a small exercise for the student to try.",
    ])
    .set_style("Friendly and encouraging. Use 'beta'/'beti' casually. Keep explanations under 100 words.")
    .build()
)

# ── Pattern 3: Dynamic System Prompt (inject real-time data) ──
def build_dynamic_prompt(user_name: str, user_plan: str, current_module: str) -> str:
    """System prompt that changes based on user context."""
    return (SystemPromptBuilder()
        .set_role(f"You are a learning assistant for {user_name}, currently on the {user_plan} plan.")
        .set_knowledge(f"""
Current progress: {current_module}
Plan: {user_plan} ({"full access" if user_plan == "pro" else "limited to Modules 3-5"})
Tip: If they ask about advanced topics beyond their plan, mention the upgrade option.""")
        .set_style("Personalized, encouraging. Reference their progress.")
        .build()
    )

print("Support bot prompt:")
print(support_prompt[:300] + "...\n")

print("Dynamic prompt for a specific user:")
print(build_dynamic_prompt("Rahul", "free", "Module 5: Prompt Engineering"))

# Output:
#   Support bot prompt:
#   # ROLE
#   You are Asha, a customer support agent for Netsetos EdTech platform.
#   # RULES (never violate)
#   1. Only answer questions about Netsetos courses, pricing, and technic...
#
#   Dynamic prompt for a specific user:
#   # ROLE
#   You are a learning assistant for Rahul, currently on the free plan.
#   # KNOWLEDGE BASE
#   Current progress: Module 5: Prompt Engineering
#   Plan: free (limited to Modules 3-5)

#### 💡 What just happened

What just happened?
  We built a `SystemPromptBuilder` with chainable methods for: role, rules, style, knowledge base, and few-shot examples. Three patterns: (1) **Customer support** - strict rules about what to discuss, with example interactions. (2) **Coding tutor** - code-first teaching style with Indian analogies. (3) **Dynamic prompts** - system prompt changes based on user data (name, plan, progress). The dynamic pattern is how SaaS products personalize AI responses per user. **When to use each:** a static string for single-purpose bots; the builder when prompts need versioning and review; dynamic injection when responses must reflect per-user state - the tradeoff is prompt caching (7.3): a prompt that changes per user cannot share a cached prefix.

---

## 🎯 Concept 3: Context Window Strategies - When History Gets Too Long

### Context Window Strategies - When History Gets Too Long

After 50 turns, your history is 30K tokens. It won't fit. Here's how to handle it.

Lesson 7.3 warned you that input tokens are where chat apps bleed money; this step is where the bleeding happens. Re-sending everything means turn 50 pays for turns 1-49 again - roughly *n(n+1)/2* cumulative growth. You need a policy for what to drop, what to compress, and what to keep verbatim - and each policy loses something different.

> 📚 **Analogy**
>
> **Think of a notebook with a fixed number of pages.** You've filled 100 pages with conversation notes. The notebook only fits 80. You have 3 options: (1) rip out the oldest 20 pages (sliding window), (2) summarize the first 60 pages into 5 pages and keep the last 15 verbatim (summarize + recent), (3) distill the key facts onto one index card taped inside the cover - names, decisions, preferences (entity memory).
> **Where this analogy breaks down:** ripping pages out of a paper notebook is forever. In your app the FULL transcript still sits safely in the session store - you only trim the *copy you send*. The model sees the abridged edition; your database keeps the director's cut. (Preview: in Module 8, RAG replaces the index card with a searchable library - retrieval instead of memorization.)

**📝 `part3_memory_strategies.py`** - *language-python*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


- **rough_tokens** - trim decisions use a free local estimate (~4 chars/token). The old version called the count-tokens API once per message per trim - N network round trips before every request. Estimate locally; bill from `response.usage_metadata`.

- **summarize_and_keep** - pays ONE cheap LLM call to compress old turns. The summary comes back as a second system-role message: read the TRAP comment - a call site that filters for user/assistant roles silently throws it away.

- **trim** - raises on unknown strategies instead of silently returning the untrimmed list (fail loud: a typo in "summarize" should not quietly re-send 30K tokens).

*SDK note:* this uses the current [google-genai SDK](https://github.com/googleapis/python-genai) (`from google import genai`); its predecessor google-generativeai reached end-of-life 2025-11-30.

#### 💡 What just happened

What just happened?
  A `MemoryManager` with 3 strategies: (1) **Sliding window** - drops oldest messages until it fits (fast, loses early context). (2) **Summarize + keep recent** - AI summarizes old messages into bullet points, keeps last 6 verbatim (best balance). (3) **Entity memory** - extracts key facts (name, goals, decisions) into a persistent profile. Strategy 2 is the production standard - users keep their recent conversation in full detail while earlier context is preserved as a concise summary.

- Scene 1: a chat with NO memory - watch the bot fumble the follow-up question (the cold-open bug, animated).

- Scene 2: every turn re-sends the whole history - follow the token counter and the cumulative-cost curve bending upward toward turn 50.

- Scene 3: the same 50-message conversation trimmed three ways side by side - sliding window drops oldest, summarize compresses them into one amber block, entity memory distills an index card.

- Scene 4: the tradeoff bars - tokens sent vs information preserved vs extra LLM calls. There is no free lunch, only chosen losses.

Controls: Play/Pause, Reset, and speed (0.5x / 1x / 2x). Use the scene buttons to jump; each scene narrates itself in the caption bar.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Persistent Storage - Survive Server Restarts

### Persistent Storage - Survive Server Restarts

In-memory sessions vanish when the server restarts. Store them in Redis or a database.

Everything so far lives in a Python dict - which means everything so far dies with the process. Every deploy, crash, or autoscaler scale-to-zero wipes every conversation mid-sentence. Persistence is what turns "a demo that works on my machine" into "a service users can come back to."

> 🗄️ **Analogy**
>
> **Whiteboard, filing cabinet, basement archive.** The in-memory dict is the office whiteboard: instant to read, wiped by the night cleaner (restart). Redis is the filing cabinet by the desk: organized folders (keys), each with an auto-shred date (TTL) so stale folders clear themselves out. PostgreSQL is the basement archive: slower to walk to, but permanent - that's where compliance and analytics live. Production chat uses the cabinet for the hot path and sends copies to the basement asynchronously.
> **Where this analogy breaks down:** a filing cabinet is slow; Redis is not - it keeps everything in RAM (with disk snapshots for durability), which is why reads are typically sub-millisecond. The cabinet metaphor explains durability and TTL, not speed. And unlike a cabinet, Redis serves a thousand hands reaching in at once.

**📝 `part4_persistent_sessions.py`** - *language-python*

In [ ]:
# =============================================
# PERSISTENT SESSIONS: Redis (async client)
# Sessions survive server restarts.  pip install redis
# =============================================

import json
import redis.asyncio as redis          # async client - matches async FastAPI
from datetime import datetime

class PersistentSessionStore:
    """Chat sessions stored in Redis - survive restarts."""

    def __init__(self, redis_url: str = "redis://localhost:6379", ttl_hours: int = 24):
        self.redis = redis.from_url(redis_url, decode_responses=True)
        self.ttl = ttl_hours * 3600

    def _key(self, session_id: str) -> str:
        return f"chat:session:{session_id}"

    async def save(self, session: ChatSession) -> None:
        data = {
            "session_id": session.session_id,
            "messages": [{"role": m.role, "content": m.content, "ts": m.timestamp}
                         for m in session.messages],
            "created": session.created_at.isoformat(),
            "updated": datetime.now().isoformat(),
        }
        await self.redis.setex(self._key(session.session_id), self.ttl, json.dumps(data))

    async def load(self, session_id: str) -> ChatSession | None:
        raw = await self.redis.get(self._key(session_id))
        if raw is None:
            return None
        data = json.loads(raw)
        session = ChatSession(session_id=data["session_id"])
        session.created_at = datetime.fromisoformat(data["created"])   # restore, don't reset
        for m in data["messages"]:
            session.messages.append(Message(role=m["role"], content=m["content"],
                                            timestamp=m.get("ts", 0)))
        return session

    async def list_sessions(self) -> list[str]:
        # SCAN, never KEYS: KEYS walks every key in one blocking O(N) sweep -
        # it freezes the whole Redis server in production. SCAN iterates in
        # small batches the server can breathe between.
        ids: list[str] = []
        async for key in self.redis.scan_iter(match="chat:session:*"):
            ids.append(key.split(":")[-1])
        return ids

# Why Redis for chat sessions?
#   - Fast: typically sub-millisecond reads (in-memory, RAM-speed)
#   - TTL: sessions auto-expire (the filing cabinet's auto-shred date)
#   - Persistent: survives restarts with AOF/RDB snapshots
# For PERMANENT history (analytics, compliance): also archive to PostgreSQL
# asynchronously. Redis = hot path, database = basement archive.

# Output: (demo with a local Redis, then a simulated restart)
#   saved  chat:session:a3f8c2e1  ttl=86400s
#   -- server restarted --
#   loaded a3f8c2e1: 3 messages, created 2026-07-02T14:03:11


#### 💡 What just happened

What just happened?
  Sessions now live in Redis with a 24-hour auto-expiry: `save()` serializes the transcript with `setex` (write + TTL in one atomic call), `load()` rebuilds a `ChatSession` - including its original `created_at` - and `list_sessions()` iterates with `SCAN` because `KEYS` freezes the whole server while it walks every key. The client is `redis.asyncio`, matching the async FastAPI world it will live in. Restarts, deploys, and autoscaling no longer end conversations mid-sentence. **When to use what:** Redis alone for ephemeral chat; Redis + PostgreSQL when analytics or compliance demand permanence; below a few hundred concurrent users a plain database with no Redis is a perfectly good alternative - simpler beats faster until it does not.

- Scene 1: a restart wipes every in-memory session card - the returning user gets an empty chat.

- Scene 2: the same crash with Redis - cards survive with TTL clocks ticking; watch one expire when its clock hits zero.

- Scene 3: the summary-discard bug - the amber summary block either falls at the role filter (red X, model never sees it) or is merged into the system instruction (green check). This is the exact bug we fixed in step 10.

- Scene 4: two-tier storage - the hot path reads Redis, a background task archives to PostgreSQL.

Controls: Play/Pause, Reset, speed. Scene 3 is the one to replay until the wrong-vs-right path is burned in.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Production-Grade: Structure, Streaming, and Shipping

Steps 1-4 gave you a working chat engine. Steps 5-9 turn it into a service you can defend in a design review: a layout that scales past one file, a streaming endpoint, logs you can query at 3 AM, a deploy pipeline, and a hardening checklist. Step 10 assembles all of it into the final project.

---

## 🎯 Concept 5: Project Structure & Dependency Injection - Production FastAPI Layout

### Project Structure & Dependency Injection - Production FastAPI Layout

Domain modules. Thin routes. Service layer. Annotated dependencies.

The single-file app you've been building is perfect for learning and painful for teams. The moment two engineers touch it, or one endpoint needs testing without the real database, you need seams: where does routing end and business logic begin? Where do shared resources live? FastAPI's answer is dependency injection, and the community's answer to layout is the domain-module pattern.

> 👨‍🍳 **Analogy**
>
> **A restaurant kitchen with stations.** A one-cook kitchen does everything at one counter - fine for a food stall, chaos at scale. A real kitchen has stations: prep, grill, plating (domain modules - `auth/`, `chat/`, `llm/`). The waiter (your route function) never cooks: they take the order, validate it, deliver the plate (thin routes - HTTP in, HTTP out). Ingredients are DELIVERED to each station before service starts (`Depends()` - the station never runs to the market mid-order). And the tandoor is lit once at opening and shared all night (the lifespan-singleton LLM client on `app.state`) - re-lighting it per kebab is how you get a new TCP+TLS handshake on every request.
> **Where this analogy breaks down:** kitchen stations are physical and expensive to rearrange; Python modules are free to refactor. Don't over-partition a 3-endpoint app into 12 folders - the pattern earns its keep from roughly the third feature onward.

> ✅ **Info**
>
> 📁 Production project structure
> - **Domain modules:** Organize by feature (auth/, chat/, llm/, health/) not by file type. Each module has its own router.py, schemas.py, models.py, service.py, dependencies.py, exceptions.py.
> - **Thin routes:** Endpoints handle HTTP only (validation, status codes). All LLM logic goes in the service layer - framework-agnostic and independently testable.
> - **Dependency injection:** `Depends(get_db)` for database sessions (auto-commit/rollback). `Depends(get_llm_client)` for LLM clients (lifespan singletons on app.state). `Annotated[AsyncSession, Depends(get_db)]` for clean type aliases.
> - **Pydantic v2:** `ConfigDict(str_strip_whitespace=True, extra="forbid")`. `field_validator` replaces `@validator`. `model_validator` for cross-field validation ("last message must be from user").
> - **Auto-docs:** FastAPI generates Swagger/OpenAPI at `/docs`. Customize with tags_metadata, endpoint descriptions, response models. SSE endpoints need manual documentation.

**📝 `minimal_di.py`** - *language-python*

In [ ]:
# minimal_di.py - the two patterns that carry every FastAPI AI service
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import Depends, FastAPI, Request

@asynccontextmanager
async def lifespan(app: FastAPI):
    from google import genai
    app.state.llm = genai.Client()      # created ONCE at startup
    print("startup: LLM client ready")
    yield
    print("shutdown: flush logs, close pools here")

app = FastAPI(lifespan=lifespan)

def get_llm(request: Request):
    return request.app.state.llm        # the singleton, injected

LLMClient = Annotated[object, Depends(get_llm)]   # clean reusable alias

@app.get("/models")
async def list_models(llm: LLMClient):
    # this route never BUILT a client - it was delivered to the station
    return {"client": type(llm).__name__}

# Output: uvicorn minimal_di:app
#   startup: LLM client ready
#   GET /models  ->  {"client": "Client"}
#   ^C  shutdown: flush logs, close pools here


- **lifespan** - everything before `yield` runs once at startup; everything after runs once at shutdown. The LLM client born here lives on `app.state` for the whole process.

- **get_llm + Depends** - routes declare WHAT they need, not HOW to build it. Tests later swap this one function (`dependency_overrides`) to inject a fake - no monkey-patching the route.

- **Annotated alias** - `LLMClient` makes the endpoint signature read like documentation: `async def chat(llm: LLMClient)`.

*Tradeoff:* DI adds a layer of indirection a 30-line script doesn't need. It pays for itself the first time you write a test or a second engineer joins.

#### 💡 What just happened

What just happened? Production FastAPI layout follows the **"domain module" pattern** (the ~16K-star [fastapi-best-practices repo](https://github.com/zhanymkanov/fastapi-best-practices)). Each feature is self-contained. Dependencies are cached within a request - calling `get_db()` twice in one request returns the same session. **LLM clients must be lifespan singletons**, not per-request - creating new HTTP connections wastes resources. Pydantic v2 is 4-50× faster than v1 (typically ~17×, per Pydantic’s own benchmarks) thanks to its Rust core. **Annotated type aliases** make endpoints clean: `async def chat(db: DbSession, user: CurrentUser, llm: LLMClient)`.

---

## 🎯 Concept 6: SSE Streaming Endpoint - fastapi.sse + Async Generators

### SSE Streaming Endpoint - fastapi.sse + Async Generators

Real-time token streaming. Unified stream/non-stream endpoint. Client disconnect detection.

Lesson 7.2 built the first leg of the relay - provider to your server. This step is the second leg: your server to the browser, as an SSE endpoint. Since FastAPI 0.135 this no longer needs a third-party package: `fastapi.sse` ships `EventSourceResponse` natively, with the keep-alive ping and anti-buffering headers handled for you.

> 🏃 **Analogy**
>
> **The second runner in a relay.** Your server receives the baton (each token) from the provider and must pass it on THE MOMENT it lands in your hand - holding batons to pass them in a bundle is exactly the buffering bug from 7.2. And `request.is_disconnected()` is the glance down the track before you sprint: if the next runner walked off (user closed the tab), you stop running - every stride after that glance is tokens you pay for that nobody will ever read.
> **Where this analogy breaks down:** a relay has one baton; you pass hundreds per response. And unlike a race, the SECOND leg can be slower than the first with no penalty - the browser renders at reading speed. What you cannot do is stop mid-leg silently: send a final done event (or an SSE error event) so the frontend knows the difference between "finished" and "died".

> 📦 **Info**
>
> 🚀 SSE streaming from FastAPI
> - **Native (FastAPI >= 0.135):** `from fastapi.sse import EventSourceResponse`. Yield strings, dicts, or `ServerSentEvent` objects; the framework sends a keep-alive ping every 15s and sets `Cache-Control: no-cache` + `X-Accel-Buffering: no` for you (the 7.2 nginx fix, now a default).
> - **Manual (any version):** `StreamingResponse(gen(), media_type="text/event-stream")` - you format every frame yourself as `data: {...}\n\n` (the blank line ends the frame) and set the headers by hand. The `sse-starlette` package pioneered this API before FastAPI absorbed it.
> - **Client disconnect:** check `await request.is_disconnected()` before each chunk - stops wasted compute when users close the tab.
> - **Unified endpoint:** one `POST /chat` with `stream: bool` in the body, returning either JSON or the event stream (the practice lab builds this variant).
> - **End signal:** a final `done` event (or OpenAI's `data: [DONE]` convention) - the frontend must be able to tell "finished" from "died".

**📝 `sse_minimal.py`** - *language-python*

In [ ]:
# sse_minimal.py - native SSE since FastAPI 0.135
# pip install "fastapi>=0.135" uvicorn
import asyncio, json
from fastapi import FastAPI, Request
from fastapi.sse import EventSourceResponse, ServerSentEvent

app = FastAPI()

@app.post("/chat/stream")
async def stream(request: Request):
    async def gen():
        tokens = ["The", " illusion", " of", " memory", " is", " engineered."]
        for token in tokens:                    # stand-in for the LLM stream
            if await request.is_disconnected():
                return                          # tab closed: stop paying
            yield ServerSentEvent(data=json.dumps({"t": token}))
            await asyncio.sleep(0.2)
        yield ServerSentEvent(data="[DONE]")
    return EventSourceResponse(gen())

# Output: curl -N -X POST localhost:8000/chat/stream
#   data: {"t": "The"}
#
#   data: {"t": " illusion"}
#   ...
#   data: [DONE]
#   (every frame ends with a blank line - the \n\n terminator from 7.2)


#### 💡 What just happened

What just happened? SSE is the standard transport for AI chat - it's what ChatGPT-class UIs use. The **unified endpoint pattern** (one route, stream flag) is cleaner than separate /chat and /chat/stream routes. **Critical gotcha (manual mode):** `X-Accel-Buffering: no` must be set per-response - without it, nginx buffers the whole response into one burst; the native `EventSourceResponse` sets it for you ([FastAPI SSE docs](https://fastapi.tiangolo.com/tutorial/server-sent-events/)). And keep the endpoint truly async end to end: a sync LLM client inside an async route measurably loses ~97% of throughput ([FastAPI discussion #10935](https://github.com/fastapi/fastapi/discussions/10935)).

- Scene 1: a SYNC LLM call freezes the single conveyor - queued requests pile up red; throughput collapses to a few requests per second.

- Scene 2: the async version - during the network wait the belt keeps serving others; compare the RPS bars (the #10935 measurement, animated).

- Scene 3: the anatomy of one chat request - route, validation, Depends() lighting up the singletons, service layer, provider, and SSE frames streaming back token by token.

- Scene 4: the vanished user - without is_disconnected() the server keeps generating for a closed tab (red wasted-token counter); with the check, generation stops early.

Controls: Play/Pause, Reset, speed. Watch scenes 1 and 2 back to back - the belt is the whole argument for async.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Observability - structlog, Langfuse, Request Tracing

### Observability - structlog, Langfuse, Request Tracing

Structured JSON logs. Per-request context. LLM cost tracking. Health checks.

At 3 AM, "the chat is slow" is not a bug report - it's a riddle. Which user? Which provider call? How many tokens? Was it the LLM or the database? Observability is the discipline of being able to answer those questions from records you set up BEFORE the incident, and for LLM apps it has one extra dimension traditional APM lacks: cost per request.

> ✈️ **Analogy**
>
> **A flight recorder and a control tower.** structlog is the black box: every event written as structured, queryable data - not prose - so after an incident you can replay exactly what happened. Langfuse is the control tower dashboard: live position, cost, and latency of every flight (request) currently in the air. The X-Request-ID header is the flight number that ties the gate log, the radar track, and the black box together across systems. And health checks are the pre-takeoff checklist: the tower (Cloud Run's load balancer) will not route passengers onto a plane that hasn't confirmed `/health/ready`.
> **Where this analogy breaks down:** a black box is only read after a crash. Structured logs earn their cost DAILY - product questions ("which feature burns the most tokens?") read the same records as incident response. If you log it well once, everyone queries it forever. Langfuse deepens into your evaluation backbone in Module 14 (LLMOps).

> ✅ **Info**
>
> 📈 Production observability stack
> - **structlog:** JSON logging with contextvars. Bind request_id, user_id at middleware level → automatically appears in all logs. Log LLM calls with latency_ms, prompt_tokens, completion_tokens, estimated_cost.
> - **Langfuse:** `@observe()` decorator on LLM functions. Tracks cost, latency, token usage per trace. Zero added latency (async background). `litellm.success_callback = ["langfuse"]` for zero-code LiteLLM tracing.
> - **Health checks:** `/health` (liveness - is process alive?). `/health/ready` (readiness - DB connected? Redis up? LLM API reachable?). Cloud Run uses these for routing decisions.
> - **X-Request-ID:** Generate UUID per request in middleware. Return in response header. Enables end-to-end tracing from frontend to logs.

**📝 `observability.py`** - *language-python*

In [ ]:
# observability.py - structured logs with per-request context
# pip install structlog
import time, uuid, structlog
from fastapi import FastAPI, Request

structlog.configure(processors=[
    structlog.contextvars.merge_contextvars,   # carries request_id everywhere
    structlog.processors.add_log_level,
    structlog.processors.TimeStamper(fmt="iso"),
    structlog.processors.JSONRenderer(),
])
logger = structlog.get_logger()
app = FastAPI()

@app.middleware("http")
async def trace_requests(request: Request, call_next):
    rid = str(uuid.uuid4())[:8]
    structlog.contextvars.bind_contextvars(request_id=rid)   # sticks to every
    t0 = time.perf_counter()                                 # log line below
    response = await call_next(request)
    logger.info("request_done", path=request.url.path,
                ms=round((time.perf_counter() - t0) * 1000, 1))
    response.headers["X-Request-ID"] = rid    # the flight number, returned
    return response

# After every LLM call, log the numbers that answer 3 AM questions:
#   logger.info("llm_call", model=MODEL, ms=latency_ms,
#               tokens_in=usage.prompt_token_count,
#               tokens_out=usage.candidates_token_count)

# Output: one queryable JSON line per event
#   {"event": "llm_call", "request_id": "a3f8c2e1",
#    "model": "gemini-2.5-flash", "ms": 842.3,
#    "tokens_in": 412, "tokens_out": 58,
#    "level": "info", "timestamp": "2026-07-02T14:03:11Z"}


#### 💡 What just happened

What just happened?**structlog + Langfuse** is the recommended observability stack. structlog produces queryable JSON logs. Langfuse tracks LLM-specific metrics (cost per trace, user, feature). Together they answer: "Whose conversation cost the most last hour?" and "Why was this one request so slow?" **Health checks** are non-negotiable for container deployments - without /health/ready, Cloud Run routes traffic to instances that can't serve (no DB connection yet). **The tradeoff:** every log line costs storage and microseconds - log request-level events and LLM calls, not every streamed token chunk; sample verbose traces when volume grows.

---

## 🎯 Concept 8: Docker & Cloud Run - Ship Your Chat API

### Docker & Cloud Run - Ship Your Chat API

Multi-stage Docker. Cloud Run SSE streaming. Secrets management. Graceful shutdown.

It works on your laptop. Docker's job is to make "your laptop" a reproducible artifact; Cloud Run's job is to run that artifact with autoscaling, HTTPS, and native SSE support - while you keep three promises: the image stays small, the secrets stay out of it, and shutdowns finish the sentences they started.

> 📦 **Analogy**
>
> **Moving house with a shipping container.** A multi-stage build packs the furniture and leaves the workshop behind - the compilers and build caches that made the furniture don't ship (image drops from gigabytes to a couple hundred MB). Your valuables travel with YOU, never packed in the container: secrets are injected at deploy time (`--set-secrets` from Secret Manager), never baked into an image layer any registry reader can extract. The new landlord won't hand over keys until the inspection passes (readiness checks gate traffic). And closing time matters: exec-form `CMD` hands SIGTERM directly to uvicorn so in-flight streams finish - shell-form wraps it in `/bin/sh`, which swallows the signal, and 10 seconds later the process is killed mid-sentence.
> **Where this analogy breaks down:** you move house once; you deploy weekly. The container's real payoff is repetition - the 40th deploy is as boring as the first. Module 12 goes deep on the full production-deployment story (CI/CD, canary, rollback); this step ships one honest service.

| Platform | SSE Streaming | Pricing | Best For |
|---|---|---|---|
| Cloud Run | Native support | Pay per use | Production + scale |
| Render | ✅ Works | from about $7/mo fixed | Predictable costs |
| Railway | ✅ Works | Usage-based | Fastest prototyping |

> 💡 **Info**
>
> 📦 Docker + Cloud Run essentials
> - **Dockerfile:** Multi-stage from `python:3.12-slim`. Use `uv` for dramatically faster installs (10-100× per Astral's own benchmarks). Exec-form CMD: `CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0"]` (shell form breaks SIGTERM).
> - **Cloud Run SSE:** Works natively over the default HTTP/1.1. Request timeout defaults to 300s and extends to 3600s (60 min) - set it above your longest stream, and make streams over 15 min reconnect-tolerant. Min instances ≥1 to avoid cold starts. Only enable `--use-http2` if your server actually speaks h2c.
> - **Secrets:** `--set-secrets="OPENAI_API_KEY=openai-key:latest"` from Secret Manager. Never bake into Dockerfile ENV.
> - **Graceful shutdown:** Lifespan pattern (`@asynccontextmanager`) closes DB engines, HTTP clients, flushes Langfuse. Set `--timeout-graceful-shutdown 30`.
> - **Config:** `pydantic-settings` with `BaseSettings(env_file=".env")`. `repr=False` on API key fields prevents log leakage.

**📝 `Dockerfile`** - *language-dockerfile*

In [ ]:
%%writefile Dockerfile
# Dockerfile - multi-stage: pack the furniture, leave the workshop
FROM python:3.12-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv
WORKDIR /app
COPY requirements.txt .
RUN uv pip install --system --no-cache -r requirements.txt

FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /usr/local/lib/python3.12/site-packages /usr/local/lib/python3.12/site-packages
COPY --from=builder /usr/local/bin/uvicorn /usr/local/bin/uvicorn
COPY app/ app/
# exec-form CMD: uvicorn receives SIGTERM directly -> graceful shutdown.
# Shell form (CMD uvicorn ...) wraps it in /bin/sh, which eats the signal.
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]

# Deploy - secrets injected at deploy time, never baked into the image:
#   gcloud run deploy chat-api --source . --region asia-south1 \
#     --set-secrets="GOOGLE_API_KEY=google-key:latest" \
#     --timeout 3600 --min-instances 1
# Output:
#   Building using Dockerfile and deploying container to Cloud Run...
#   Service [chat-api] revision [chat-api-00001-abc] has been deployed
#   Service URL: https://chat-api-xxxxx.asia-south1.run.app


#### 💡 What just happened

What just happened? Cloud Run is the recommended deployment for AI chat - it supports SSE natively, scales to zero, and allows request timeouts up to 3600s ([Cloud Run request-timeout docs](https://docs.cloud.google.com/run/docs/configuring/request-timeout)). **Do NOT use the old tiangolo/uvicorn-gunicorn-fastapi image** - it's deprecated. Use `python:3.12-slim` with multi-stage builds. **Exec-form CMD is critical:** shell form wraps in /bin/sh which doesn't forward SIGTERM, causing 10s forced kills instead of graceful shutdown. Render (from about $7/mo) and Railway (usage-based) are simpler alternatives for smaller projects.

- Scene 1: the multi-stage build - the workshop (build tools) stays behind and the shipped image shrinks from gigabytes to a couple hundred MB.

- Scene 2: secrets - the wrong way bakes the key into a readable image layer; the right way injects it at deploy from Secret Manager.

- Scene 3: health checks - instances receive traffic only after /health/ready goes green; watch the load balancer route around a failing one.

- Scene 4: shutdown - exec-form CMD lets in-flight streams finish; shell-form swallows SIGTERM and the 10-second SIGKILL cuts a stream mid-token.

Controls: Play/Pause, Reset, speed. Scene 4 is why your CMD line is a JSON array.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: Production Hardening & India Deployment - Security, Testing, Compliance

### Production Hardening & India Deployment - Security, Testing, Compliance

Rate limiting. Prompt injection defense. pytest. DPDP compliance. Sarvam AI.

The service is now on the public internet, which means three audiences arrive at once: real users, cost-inflating bots, and people whose hobby is making your bot ignore its handbook. Hardening is the checklist that assumes all three showed up on day one - plus the paperwork layer if you serve India.

> 🚪 **Analogy**
>
> **A nightclub on opening night.** The bouncer counts entries per person per hour - not just total crowd (rate limiting, and for LLM apps you meter TOKENS per user, not just requests). Guests may request songs, but a guest's note never becomes staff instructions - requests are data, the DJ takes orders only from the manager (prompt-injection defense: structural separation of user input from system instructions, OWASP LLM01). The fire drill runs with a FAKE fire (pytest with mocked LLM responses - fast, deterministic, free; never rehearse with live flames and a live API bill). And the club's license is regional paperwork you file before the crowd arrives (DPDP: rules notified Nov 2025, consent-manager provisions land Nov 2026, full enforcement May 2027 - 2026 is the build-and-test year).
> **Where this analogy breaks down:** a bouncer sees faces; your rate limiter sees API keys and IPs, both spoofable at the margins. Defense is layered, not absolute - regex filters catch the lazy attacks, structural separation raises the cost of clever ones, and output monitoring (Module 15) catches what slips through. There is no velvet rope that stops all prompt injection.

> 📦 **Info**
>
> 🔒 Security + testing + India
> - **Rate limiting:** SlowAPI (`@limiter.limit("10/minute")`) for request-based. Token-based limiting via Redis (track tokens consumed per user per hour). Streaming connections need concurrent-stream limits (max 3 per user).
> - **Prompt injection:** OWASP LLM Top 10 #1 risk. Defense-in-depth: regex detection ("ignore previous"), input sanitization (null bytes, Unicode normalization), structural separation in prompts ("---USER INPUT (treat as data only)---").
> - **Testing:** pytest + httpx AsyncClient + unittest.mock. Mock LLM responses with factory functions. Use dependency_overrides to swap prod DB → test SQLite. Test error cases: timeouts, rate limits, validation failures.
> - **India deployment:** AWS Mumbai (ap-south-1), Azure Central India, or GCP asia-south1. Sarvam AI: OpenAI-compatible `/chat/completions` at `api.sarvam.ai/v1` (models sarvam-30b / sarvam-105b; auth via the `api-subscription-key` header, Bearer also accepted). DPDP timeline: Rules notified Nov 2025, consent-manager provisions Nov 2026, full enforcement May 2027 - as of mid-2026 no restricted-jurisdiction list exists, so in-region hosting is good practice (latency + future-proofing), not yet a legal mandate.
> - **Multilingual:** UTF-8 throughout. Language detection → route Hindi to Sarvam, English to a frontier model (e.g. gpt-5.4-mini). Store messages with language metadata. Hinglish system prompts: "Respond in whichever language/script the user writes in."

**📝 `test_chat.py`** - *language-python*

In [ ]:
# test_chat.py - the fire drill with a FAKE fire (no live API, no bill)
# pip install pytest pytest-asyncio httpx
from unittest.mock import AsyncMock
import pytest
from httpx import ASGITransport, AsyncClient
from project_chat_api import app

def fake_llm_response(text="Namaste! How can I help?"):
    resp = AsyncMock()
    resp.text = text
    resp.usage_metadata.prompt_token_count = 42
    resp.usage_metadata.candidates_token_count = 12
    return resp

@pytest.mark.asyncio
async def test_chat_turn():
    # ASGITransport skips lifespan - install the fake singleton yourself
    app.state.llm = AsyncMock()
    app.state.llm.aio.models.generate_content = AsyncMock(
        return_value=fake_llm_response())
    async with AsyncClient(transport=ASGITransport(app=app),
                           base_url="http://test") as http:
        r = await http.post("/sessions", json={})
        sid = r.json()["session_id"]
        r = await http.post(f"/sessions/{sid}/chat",
                            json={"message": "Hello!"})
    assert r.status_code == 200
    assert r.json()["turn"] == 1

# Also test the error paths - that's where production failures live:
#   mock raising a timeout  -> expect 503 AND the user turn rolled back
#   provider returns a 429  -> expect your handler's Retry-After header

# Output: pytest -q
#   .                                                        [100%]
#   1 passed in 0.31s


#### 💡 What just happened

What just happened? Production hardening covers **three layers:** security (rate limiting + prompt injection defense), quality (pytest with mocked LLM responses), and compliance (DPDP for India). **Sarvam AI integrates as a drop-in:** point the OpenAI SDK at base_url api.sarvam.ai/v1 (auth via the api-subscription-key header; Bearer also accepted) - same code, Indian data residency, ₹ billing, strong Indic-language support (models sarvam-30b / sarvam-105b). For testing: never call real LLM APIs in tests - mock responses with factory functions that return realistic token counts and content. **SQLite for dev, PostgreSQL for prod** - switch via ENVIRONMENT variable in pydantic-settings. **When to harden how much:** a closed beta can ship with rate limits alone; anything public needs the full checklist - the alternative is meeting your API again on a bot farm’s invoice.

---

## 🎯 Concept 10: Project: Production Chat API - Everything, Integrated

### Project: Production Chat API - Everything, Integrated

Sessions + memory + streaming + persistence - all in one deployable FastAPI app.

Rehearsals are over. This is every previous step on stage at once: the session store (step 1), the prompt builder (step 2), memory trimming that is *actually sent to the model* (step 3), async everything (steps 5-6), and native SSE with a disconnect guard (step 6). Compare it against the naive version from the cold open as you read - every difference is a scar from a real bug.

> 🎭 **Analogy**
>
> **Opening night.** The front desk checks you in (session CRUD), the prompter holds the script (system prompt), the editor has already cut act one to fit the evening (memory trim - and unlike our first draft, the cut version is what the actors actually perform), and the show streams live to every seat (SSE). One company, one stage, no missing departments.

**📝 `project_chat_api.py`** - *language-python*

In [ ]:
# =============================================
# PRODUCTION CHAT API - the honest version
# Sessions + Memory + Streaming + Persistence
# pip install "fastapi>=0.135" uvicorn google-genai
# Reuses the classes you built in steps 1-3:
from part1_session_manager import Message, SessionStore
from part2_system_prompts import SystemPromptBuilder
from part3_memory_strategies import MemoryManager
# =============================================

import asyncio, json
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.sse import EventSourceResponse, ServerSentEvent   # FastAPI 0.135+
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.llm = genai.Client()   # ONE client for the app's lifetime -
    yield                            # a per-request client pays a TCP+TLS
                                     # handshake tax on every single call

app = FastAPI(title="Netsetos Chat API", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://app.example.com"],   # never "*" in production
    allow_methods=["GET", "POST", "DELETE"],
    allow_headers=["*"],
)

MODEL = "gemini-2.5-flash"
sessions = SessionStore()
memory = MemoryManager(max_history_tokens=8000)

DEFAULT_SYSTEM = (SystemPromptBuilder()
    .set_role("You are a helpful AI tutor for the Netsetos GenAI course.")
    .set_rules(["Keep answers under 150 words.", "Use code examples when helpful."])
    .build())

class CreateSession(BaseModel):
    # Client-supplied system prompts are an injection surface (step 9):
    # cap the size; in production prefer an allow-list of named templates.
    system_prompt: str = Field(default=DEFAULT_SYSTEM, max_length=4000)

class ChatMessage(BaseModel):
    message: str = Field(min_length=1, max_length=8000)

def prepare_context(session) -> tuple[str, list[types.Content]]:
    """Trim history, PERSIST the compaction, split system text from turns."""
    api_messages = memory.trim(session.to_api_format(), strategy="summarize")
    if len(api_messages) < len(session.messages):
        # Persist the compaction - otherwise next turn re-summarizes the
        # ever-growing ORIGINAL: one wasted LLM call per turn, forever.
        session.messages = [Message(role=m["role"], content=m["content"])
                            for m in api_messages]
    # EVERY system-role message (original prompt + any summary) reaches the
    # model. This line is the fix for the silently-discarded-summary bug.
    system_text = "\n\n".join(m["content"] for m in api_messages
                               if m["role"] == "system")
    contents = [types.Content(role="user" if m["role"] == "user" else "model",
                              parts=[types.Part(text=m["content"])])
                for m in api_messages if m["role"] in ("user", "assistant")]
    return system_text, contents

@app.post("/sessions")
async def create_session(req: CreateSession):
    session = sessions.create(system_prompt=req.system_prompt)
    return {"session_id": session.session_id, "turns": 0}

@app.get("/sessions/{session_id}")
async def get_session(session_id: str):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    return {"session_id": session.session_id, "turns": session.turn_count(),
            "messages": session.to_api_format()}

@app.post("/sessions/{session_id}/chat")
async def chat(session_id: str, req: ChatMessage):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    session.add_user_message(req.message)
    try:
        # prepare_context may call the LLM to summarize (sync helper from
        # step 3) - to_thread keeps it OFF the event loop. The chat call
        # itself uses the native async client: client.aio.
        system_text, contents = await asyncio.to_thread(prepare_context, session)
        resp = await app.state.llm.aio.models.generate_content(
            model=MODEL, contents=contents,
            config=types.GenerateContentConfig(system_instruction=system_text))
    except Exception:
        session.messages.pop()   # roll back the user turn - no orphan on retry
        raise HTTPException(503, "Model call failed - please retry")
    session.add_assistant_message(resp.text)
    return {"response": resp.text, "turn": session.turn_count(),
            "tokens": {"in": resp.usage_metadata.prompt_token_count,
                       "out": resp.usage_metadata.candidates_token_count}}

@app.post("/sessions/{session_id}/chat/stream")
async def chat_stream(session_id: str, req: ChatMessage, request: Request):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    session.add_user_message(req.message)
    system_text, contents = await asyncio.to_thread(prepare_context, session)

    async def generate():
        full = ""
        stream = await app.state.llm.aio.models.generate_content_stream(
            model=MODEL, contents=contents,   # FULL trimmed history - the
            config=types.GenerateContentConfig(   # goldfish bug fix
                system_instruction=system_text))
        async for chunk in stream:
            if await request.is_disconnected():   # user closed the tab:
                break                             # stop generating, stop paying
            if chunk.text:
                full += chunk.text
                yield ServerSentEvent(data=json.dumps(
                    {"type": "token", "content": chunk.text}))
        if full:
            session.add_assistant_message(full)   # save even a partial answer -
        yield ServerSentEvent(data=json.dumps(     # history stays consistent
            {"type": "done", "turn": session.turn_count()}))

    return EventSourceResponse(generate())

@app.delete("/sessions/{session_id}")
async def delete_session(session_id: str):
    sessions.delete(session_id)
    return {"deleted": session_id}

# Run: uvicorn project_chat_api:app --reload
# Output: (turn 3 of a conversation - note the history-aware answer and
#          the input-token count carrying the trimmed history)
#   POST /sessions/{id}/chat  {"message": "So which one should I pick?"}
#   {"response": "For a cracked screen, choose the replacement - it ships
#     within 5 days and your order #4521 qualifies...",
#    "turn": 3, "tokens": {"in": 412, "out": 58}}
#   -- the cold-open demo would have answered: "Which one of WHAT?"


#### 💡 What just happened

What just happened?
  We built the complete production chat API - and every difference from the cold-open's naive version is a bug fix you watched happen: (1) **Async end to end** - the LLM client is a lifespan singleton and every call goes through `client.aio`; the summarize helper runs in `asyncio.to_thread` so nothing blocks the event loop. (2) **Memory that reaches the model** - both endpoints send the FULL trimmed history, every system-role message (prompt + summary) is merged into `system_instruction`, and the compaction is persisted so you never re-summarize the same turns twice. (3) **Streams that stop** - `is_disconnected()` ends generation when the tab closes, and partial answers are still saved so history stays consistent. (4) **Failure honesty** - a failed model call rolls back the user turn and returns 503; token usage comes from `usage_metadata`, not an estimator. Swap `SessionStore` for step 4's `PersistentSessionStore` and this survives restarts too.

- **prepare_context()** - the heart of the fix. Trim, persist the compaction, merge ALL system-role text, convert to typed `Content` turns. One function, four bug fixes.

- **asyncio.to_thread(...)** - step 3's summarizer is sync; running it on a worker thread keeps the event loop serving other users (the ~97%-RPS lesson from quiz 1).

- **session.messages.pop() on failure** - without the rollback, a retried request double-adds the user turn and the transcript drifts from reality.

- **if full: add_assistant_message(full)** - a disconnected stream still saves what was generated; the user's next request sees a consistent history.

*Tradeoff:* in-memory `SessionStore` keeps the file runnable standalone; production swaps in the Redis store from step 4 (one import, two awaits).

## Recap

> ℹ️ **Info**
>
> What we built
> - **ChatSession + SessionStore** - track conversations per user with proper message format, turn counting, and API-ready output.
> - **SystemPromptBuilder** - chainable builder for role, rules, style, knowledge, and examples. Three patterns: support bot, coding tutor, and dynamic (per-user) prompts.
> - **MemoryManager** - 3 strategies: sliding window (fast), summarize + keep recent (best balance), entity memory (persistent facts). Auto-trims when history exceeds token budget.
> - **PersistentSessionStore** - Redis-backed sessions with TTL. Sub-millisecond read/write. Sessions survive server restarts.
> - **Production layout** - domain modules, thin routes, and lifespan-singleton clients injected with `Depends()`; the reason your service survives its second engineer.
> - **Native SSE endpoint** - FastAPI 0.135's `fastapi.sse.EventSourceResponse` with client-disconnect detection, so abandoned streams stop billing you.
> - **Observability** - structlog JSON logs + Langfuse traces + X-Request-ID; you can answer "whose conversation cost what, and why was it slow" from records, not memory.
> - **Shipping + hardening** - multi-stage Docker on Cloud Run (secrets injected, SIGTERM-graceful), SlowAPI rate limits, injection defense, pytest with mocked LLMs, DPDP timeline.
> - **Production Chat API** - FastAPI with session CRUD, non-streaming + SSE streaming endpoints, memory trimming that reaches the model, and system prompt preservation.

> ✅ **Info**
>
> 🔗 Where this goes next
> - **Lesson 7.5 - Function Calling & Tools:** your chat can talk; next it acts - checking real order status instead of apologizing gracefully.
> - **Module 8 - RAG:** entity memory's index card grows into a searchable knowledge base; retrieval replaces memorization.
> - **Module 12 - Production Deployment** and **Module 14 - LLMOps:** the deploy pipeline and the Langfuse traces you set up here become full disciplines.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Building AI Chat with FastAPI - From Stateless to Stateful**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-7_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-7.4.html` - regenerate this notebook whenever the source HTML is updated.*
